In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from os.path import join as pjoin
from tqdm.notebook import tqdm
from sklearn.metrics import mutual_info_score
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore, ttest_rel
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/reward_distance_overlap'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
excluded_mice = ['mc46', 'mc56'] ## missing data for the first day in one context, so didn't include in this analysis
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
pos_distances = np.arange(0, (reward_bin_size*4) + reward_bin_size, reward_bin_size) ## distance from reward locations
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 1 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians
fr_bin_size = 1 ## in seconds

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Look at change in distance from reward from day 14 to 15 and day 15 to 16 with cells shared across all three days.

In [ ]:
## Settings
experiment = 'MultiCon_Imaging6'
mouse = 'mc54'
crossreg_str = '14_15_16'
days_of_int = [14, 15, 16]
cell_type = 'all_cells'
correct_dir = True 
only_running = True
crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

In [ ]:
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'neuron_idx': [], 'min_rw_distance': []}
## Select cells that are cross-registered across days 14, 15, and 16
mappings_res = pd.read_pickle(pjoin(crossreg_path, f'mappings_meta_{centroid_distance}_{crossreg_str}.pkl'))['session'].dropna().reset_index(drop=True)

for idx, d in tqdm(enumerate(days_of_int)):
    session = f'{mouse}_{data_type}_{d}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
    S = xr.open_dataset(pjoin(exp_path, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    group = S.attrs['group']
    sex = S.attrs['sex']
    sdata = S.sel(unit_id=mappings_res[S.attrs['date']].to_numpy())
    sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
    if cell_type == 'place_cells':
        sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
    elif cell_type == 'nonplace_cells':
        sdata = sdata[~sdata['skaggs_place'], :]
    else:
        pass
    spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
    neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                    velocity_thresh=velocity_thresh)
    ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
    population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
    active_cells = np.sum(population_activity, axis=0) != 0
    population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
    tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
    tuning_curves = tuning_curves.T ## cells x spatial bin

    ## Get where the peak of the tuning curve is for each cell
    field_dist = pc.place_field_peak(tuning_curves, bins)

    ## Get reward positions
    reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
    reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

    ## Make Reward 1 the first rewarding port the mouse got water from
    first_rew = sdata['lick_port'][sdata['water']].values[0]
    if first_rew == sdata.attrs['reward_one']:
        first_rw_pos = reward_one_pos 
        second_rw_pos = reward_two_pos
    else:
        first_rw_pos = reward_two_pos 
        second_rw_pos = reward_one_pos

    ## Find distance from reward locations
    ## Negative values would be after the reward port because linear position decreases across a trial
    distance_from_both = np.zeros((field_dist.shape[0], 2))
    distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
    distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
    min_rw_distance = np.min(distance_from_both, axis=1)
    min_rw_distance = min_rw_distance * conversion

    for uid in np.arange(0, min_rw_distance.shape[0]):
        cell_dict['mouse'].append(mouse)
        cell_dict['group'].append(group)
        cell_dict['sex'].append(sex)
        cell_dict['day'].append(d)
        cell_dict['neuron_idx'].append(uid)
        cell_dict['min_rw_distance'].append(min_rw_distance[uid])
rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
## Example scatterplot of minimum reward distance on days 14 and 15, 15 and 16
day_pairs = [(14, 15), (15, 16)]
fig = pf.custom_graph_template(x_title='', y_title='', rows=1, columns=2, width=1000, 
                               shared_x=True, shared_y=True)

for idx, pair in enumerate(day_pairs):
    plot_data = rw_dist_df[(rw_dist_df['day'] == pair[0]) | (rw_dist_df['day'] == pair[1])]
    fig.add_trace(go.Scattergl(x=plot_data['min_rw_distance'][plot_data['day'] == pair[1]], y=plot_data['min_rw_distance'][plot_data['day'] == pair[0]],
                               mode='markers', marker_color=ce_colors_dict[plot_data['group'].unique()[0]], opacity=0.6,
                               showlegend=False, marker=dict(line=dict(width=1.5, color='black'))), row=1, col=idx + 1)
    fig.update_xaxes(title=f'Day {pair[1]} Reward Distance (cm)', col=idx + 1)
    fig.update_yaxes(title=f'Day {pair[0]} Reward Distance (cm)', col=idx + 1)
fig.add_hline(y=reward_bin_size * conversion, line_dash='dash', line_color=chance_color, line_width=2, opacity=1)
fig.add_vline(x=reward_bin_size * conversion, line_dash='dash', line_color=chance_color, line_width=2, opacity=1)
fig.show()

### Example mouse. Look at change in distance from reward from day 14 to 15 and day 15 to 16 with cells shared across pairs of days.

In [ ]:
## Settings
experiment = 'MultiCon_Imaging6'
mouse = 'mc55'
crossreg_str = None
cell_type = 'all_cells'
correct_dir = True 
only_running = True
crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'

In [ ]:
day_pairs = [(14, 15), (15, 16)]
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day_one': [], 'day_two': [], 'current_session': [], 'neuron_idx': [], 'min_rw_distance': []}
## Select cells that are cross-registered across days 14, 15, and 16
mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)

for idx, pair in tqdm(enumerate(day_pairs)):
    s1 = f'{mouse}_{data_type}_{pair[0]}.nc'
    s2 = f'{mouse}_{data_type}_{pair[1]}.nc'
    exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
    session_one = xr.load_dataset(pjoin(exp_path, s1))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
    session_two = xr.load_dataset(pjoin(exp_path, s2))[data_type]
    group = session_one.attrs['group']
    sex = session_one.attrs['sex']
    ## Select shared cells for both sessions
    shared_cells = mappings.loc[:, [session_one.attrs['date'], session_two.attrs['date']]].dropna().reset_index(drop=True)

    for idx, d in enumerate(pair):
        S = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{d}.nc'))[data_type]
        sdata = S.sel(unit_id=shared_cells[S.attrs['date']].to_numpy())
        sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
        if cell_type == 'place_cells':
            sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
        elif cell_type == 'nonplace_cells':
            sdata = sdata[~sdata['skaggs_place'], :]
        else:
            pass
        spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                        velocity_thresh=velocity_thresh)
        ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
        active_cells = np.sum(population_activity, axis=0) != 0
        population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
        tuning_curves = tuning_curves.T ## cells x spatial bin

        ## Get where the peak of the tuning curve is for each cell
        field_dist = pc.place_field_peak(tuning_curves, bins)

        ## Get reward positions
        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

        ## Make Reward 1 the first rewarding port the mouse got water from
        first_rew = sdata['lick_port'][sdata['water']].values[0]
        if first_rew == sdata.attrs['reward_one']:
            first_rw_pos = reward_one_pos 
            second_rw_pos = reward_two_pos
        else:
            first_rw_pos = reward_two_pos 
            second_rw_pos = reward_one_pos

        ## Find distance from reward locations
        ## Negative values would be after the reward port because linear position decreases across a trial
        distance_from_both = np.zeros((field_dist.shape[0], 2))
        distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
        distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
        min_rw_distance = np.min(distance_from_both, axis=1)
        min_rw_distance = min_rw_distance * conversion ## convert to centimeters

        for uid in np.arange(0, min_rw_distance.shape[0]):
            cell_dict['mouse'].append(mouse)
            cell_dict['group'].append(group)
            cell_dict['sex'].append(sex)
            cell_dict['day_one'].append(pair[0])
            cell_dict['day_two'].append(pair[1])
            cell_dict['current_session'].append(d)
            cell_dict['neuron_idx'].append(uid)
            cell_dict['min_rw_distance'].append(min_rw_distance[uid])
rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
## Example scatterplot of minimum reward distance on days 14 and 15, 15 and 16
fig = pf.custom_graph_template(x_title='', y_title='', rows=1, columns=2, width=1000, 
                               shared_x=True, shared_y=True)

for idx, pair in enumerate(day_pairs):
    xaxis = pair[1]
    yaxis = pair[0]
    plot_data = rw_dist_df[(rw_dist_df['day_one'] == pair[0]) | (rw_dist_df['day_two'] == pair[1])]
    fig.add_trace(go.Scattergl(x=plot_data['min_rw_distance'][plot_data['current_session'] == xaxis], y=plot_data['min_rw_distance'][plot_data['current_session'] == yaxis],
                               mode='markers', marker_color=ce_colors_dict[plot_data['group'].unique()[0]], opacity=0.6,
                               showlegend=False, marker=dict(line=dict(width=1.5, color='black'))), row=1, col=idx + 1)
    fig.update_xaxes(title=f'Day {xaxis} Reward Distance (cm)', col=idx + 1)
    fig.update_yaxes(title=f'Day {yaxis} Reward Distance (cm)', col=idx + 1)
fig.add_hline(y=reward_bin_size * conversion, line_dash='dash', line_color=chance_color, line_width=2, opacity=1)
fig.add_vline(x=reward_bin_size * conversion, line_dash='dash', line_color=chance_color, line_width=2, opacity=1)
fig.show()

### Select cells around reward locations on day 15 and see how much their place field peaks shift.

In [ ]:
days_of_int = [15, 16]
reference_day = 16
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': []}
## Select cells that are cross-registered across days 14, 15, and 16
mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)

exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
date_list = []
for d in days_of_int:
    session = f'{mouse}_{data_type}_{d}.nc'
    S = xr.load_dataset(pjoin(exp_path, session))[data_type]
    date_list.append(S.attrs['date'])
## Select shared cells
shared_cells = mappings[date_list].dropna().reset_index(drop=True)
ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
sdata = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{reference_day}.nc'))
sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :]
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass

neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Get where the peak of the tuning curve is for each cell
field_dist = pc.place_field_peak(tuning_curves, bins)

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

## Find distance from reward locations
rw_one_dist = field_dist - first_rw_pos
rw_two_dist = field_dist - second_rw_pos
bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
sub_uids = neural_data['unit_id'][active_cells]
sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

for idx, d in enumerate(days_of_int):
    if d != reference_day:
        S = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{d}.nc'))[data_type]
        sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()) ## select the equivalent cell from the ref day
        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                        velocity_thresh=velocity_thresh)
        ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
        active_cells = np.sum(population_activity, axis=0) != 0
        population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
        tuning_curves = tuning_curves.T ## cells x spatial bin
        ## Get where the peak of the tuning curve is for each cell
        field_dist = pc.place_field_peak(tuning_curves, bins)

        ## Get reward positions
        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

        ## Make Reward 1 the first rewarding port the mouse got water from
        first_rew = sdata['lick_port'][sdata['water']].values[0]
        if first_rew == sdata.attrs['reward_one']:
            first_rw_pos = reward_one_pos 
            second_rw_pos = reward_two_pos
        else:
            first_rw_pos = reward_two_pos 
            second_rw_pos = reward_one_pos

        ## Find distance from reward locations
        ## Negative values would be after the reward port because linear position decreases across a trial
        distance_from_both = np.zeros((field_dist.shape[0], 2))
        distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
        distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
        min_rw_distance = np.min(distance_from_both, axis=1)
        min_rw_distance = min_rw_distance * conversion ## convert to centimeters

        for idx in np.arange(0, min_rw_distance.shape[0]):
            cell_dict['mouse'].append(mouse)
            cell_dict['group'].append(S.attrs['group'])
            cell_dict['sex'].append(S.attrs['sex'])
            cell_dict['day'].append(d)
            cell_dict['cell_type'].append('reward_area')
            cell_dict['unit_id'].append(sub_uids.values[idx])
            cell_dict['min_rw_distance'].append(min_rw_distance[idx])
            cell_dict['spatial_info'].append(sdata['skaggs_info'].values[idx])
rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
days_of_int = [15, 16]
reference_day = 16
port_type = 'undershooting'
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': []}
## Select cells that are cross-registered across days 14, 15, and 16
mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)

exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/{mouse}/{data_type}')
date_list = []
for d in days_of_int:
    session = f'{mouse}_{data_type}_{d}.nc'
    S = xr.load_dataset(pjoin(exp_path, session))[data_type]
    date_list.append(S.attrs['date'])
## Select shared cells
shared_cells = mappings[date_list].dropna().reset_index(drop=True)
ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
sdata = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{reference_day}.nc'))
sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :]
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass

neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Get where the peak of the tuning curve is for each cell
field_dist = pc.place_field_peak(tuning_curves, bins)

## Get positions of the under or over-shooting ports and use those as the zero location
front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
if port_type == 'overshooting':
    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
elif port_type == 'undershooting':
    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)

## Find distance from under or overshooting ports and select uids near those ports
rw_one_dist = field_dist - first_rw_pos
rw_two_dist = field_dist - second_rw_pos
bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
sub_uids = neural_data['unit_id'][active_cells]
sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

for idx, d in enumerate(days_of_int):
    if d != reference_day:
        S = xr.load_dataset(pjoin(exp_path, f'{mouse}_{data_type}_{d}.nc'))[data_type]
        sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()) ## select the equivalent cell from the ref day
        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                        velocity_thresh=velocity_thresh)
        ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
        active_cells = np.sum(population_activity, axis=0) != 0
        population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
        tuning_curves = tuning_curves.T ## cells x spatial bin
        ## Get where the peak of the tuning curve is for each cell
        field_dist = pc.place_field_peak(tuning_curves, bins)

        ## Get reward positions
        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

        ## Make Reward 1 the first rewarding port the mouse got water from
        first_rew = sdata['lick_port'][sdata['water']].values[0]
        if first_rew == sdata.attrs['reward_one']:
            first_rw_pos = reward_one_pos 
            second_rw_pos = reward_two_pos
        else:
            first_rw_pos = reward_two_pos 
            second_rw_pos = reward_one_pos

        ## Find distance from reward locations
        ## Negative values would be after the reward port because linear position decreases across a trial
        distance_from_both = np.zeros((field_dist.shape[0], 2))
        distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
        distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
        min_rw_distance = np.min(distance_from_both, axis=1)
        min_rw_distance = min_rw_distance * conversion ## convert to centimeters

        for idx in np.arange(0, min_rw_distance.shape[0]):
            cell_dict['mouse'].append(mouse)
            cell_dict['group'].append(S.attrs['group'])
            cell_dict['sex'].append(S.attrs['sex'])
            cell_dict['day'].append(d)
            cell_dict['cell_type'].append(port_type)
            cell_dict['unit_id'].append(sub_uids.values[idx])
            cell_dict['min_rw_distance'].append(min_rw_distance[idx])
            cell_dict['spatial_info'].append(sdata['skaggs_info'].values[idx])
non_rw_dist_df = pd.DataFrame(cell_dict)

### Combine across mice.

In [ ]:
## Settings
only_running = True 
correct_dir = True
days_of_int = [15, 16] ## make reference day your second day. Asking from the cells on the second day in the list, what were they doing on the first day in the list
reference_day = 16
port_type = 'undershooting'
crossreg_str = None
cell_type = 'all_cells'

In [ ]:
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': [], 
             'firing_rate': [], 'odd_even': [], 'first_second': []}

for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        fpath = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(fpath)):
            mpath = pjoin(fpath, f'{mouse}/{data_type}')
            crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
            mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)
            
            date_list = []
            for d in days_of_int:
                session = f'{mouse}_{data_type}_{d}.nc'
                S = xr.load_dataset(pjoin(mpath, session))[data_type]
                date_list.append(S.attrs['date'])
            ## Select shared cells
            shared_cells = mappings[date_list].dropna().reset_index(drop=True)
            ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
            sdata = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{reference_day}.nc'))
            sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
            sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
            if cell_type == 'place_cells':
                sdata = sdata[sdata['skaggs_place'], :]
            elif cell_type == 'nonplace_cells':
                sdata = sdata[~sdata['skaggs_place'], :]
            else:
                pass

            neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                            velocity_thresh=velocity_thresh)
            ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
            population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
            active_cells = np.sum(population_activity, axis=0) != 0
            population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
            tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
            tuning_curves = tuning_curves.T ## cells x spatial bin

            ## Get where the peak of the tuning curve is for each cell
            field_dist = pc.place_field_peak(tuning_curves, bins)

            ## Get reward positions
            reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
            reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

            ## Make Reward 1 the first rewarding port the mouse got water from
            first_rew = sdata['lick_port'][sdata['water']].values[0]
            if first_rew == sdata.attrs['reward_one']:
                first_rw_pos = reward_one_pos 
                second_rw_pos = reward_two_pos
            else:
                first_rw_pos = reward_two_pos 
                second_rw_pos = reward_one_pos

            ## Find distance from reward locations
            rw_one_dist = field_dist - first_rw_pos
            rw_two_dist = field_dist - second_rw_pos
            bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
            rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
            rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
            sub_uids = neural_data['unit_id'][active_cells]
            sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

            for idx, d in enumerate(days_of_int):
                if d != reference_day:
                    S = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{d}.nc'))[data_type]
                    sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()) ## select the equivalent cell from the ref day
                    neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                    velocity_thresh=velocity_thresh)
                    ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                    population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                    active_cells = np.sum(population_activity, axis=0) != 0
                    population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                    tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                    tuning_curves = tuning_curves.T ## cells x spatial bin
                    ## Get where the peak of the tuning curve is for each cell
                    field_dist = pc.place_field_peak(tuning_curves, bins)

                    ## Get reward positions
                    reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                    reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                    ## Make Reward 1 the first rewarding port the mouse got water from
                    first_rew = sdata['lick_port'][sdata['water']].values[0]
                    if first_rew == sdata.attrs['reward_one']:
                        first_rw_pos = reward_one_pos 
                        second_rw_pos = reward_two_pos
                    else:
                        first_rw_pos = reward_two_pos 
                        second_rw_pos = reward_one_pos

                    ## Find distance from reward locations
                    ## Negative values would be after the reward port because linear position decreases across a trial
                    distance_from_both = np.zeros((field_dist.shape[0], 2))
                    distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
                    distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
                    min_rw_distance = np.min(distance_from_both, axis=1)
                    min_rw_distance = min_rw_distance * conversion ## convert to centimeters

                    ## Get firing rate across the session while running in the correct direction
                    act_bin = ctn.bin_activity(neural_data[active_cells].values, bin_size_seconds=fr_bin_size, func=np.mean) ## select active sells
                    avg_act = np.mean(act_bin, axis=1) / fr_bin_size ## convert to Hz for any bin size

                    for idx in np.arange(0, min_rw_distance.shape[0]):
                        cell_dict['mouse'].append(mouse)
                        cell_dict['group'].append(S.attrs['group'])
                        cell_dict['sex'].append(S.attrs['sex'])
                        cell_dict['day'].append(d)
                        cell_dict['cell_type'].append('reward_area')
                        cell_dict['unit_id'].append(sub_uids.values[idx])
                        cell_dict['min_rw_distance'].append(min_rw_distance[idx])
                        cell_dict['spatial_info'].append(sdata['skaggs_info'].values[idx])
                        cell_dict['firing_rate'].append(avg_act[idx])
                        cell_dict['odd_even'].append(sdata['odd_even'].values[idx])
                        cell_dict['first_second'].append(sdata['first_second'].values[idx])
rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': [], 
             'firing_rate': [], 'odd_even': [], 'first_second': []}

for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        fpath = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(fpath)):
            mpath = pjoin(fpath, f'{mouse}/{data_type}')
            crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
            mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)

            date_list = []
            for d in days_of_int:
                session = f'{mouse}_{data_type}_{d}.nc'
                S = xr.load_dataset(pjoin(mpath, session))[data_type]
                date_list.append(S.attrs['date'])
            ## Select shared cells
            shared_cells = mappings[date_list].dropna().reset_index(drop=True)
            ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
            sdata = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{reference_day}.nc'))
            sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
            sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
            if cell_type == 'place_cells':
                sdata = sdata[sdata['skaggs_place'], :]
            elif cell_type == 'nonplace_cells':
                sdata = sdata[~sdata['skaggs_place'], :]
            else:
                pass

            neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                            velocity_thresh=velocity_thresh)
            ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
            population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
            active_cells = np.sum(population_activity, axis=0) != 0
            population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
            tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
            tuning_curves = tuning_curves.T ## cells x spatial bin

            ## Get where the peak of the tuning curve is for each cell
            field_dist = pc.place_field_peak(tuning_curves, bins)

            ## Get positions of the under or over-shooting ports and use those as the zero location
            front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
            if port_type == 'overshooting':
                first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
                second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
            elif port_type == 'undershooting':
                first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
                second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)

            ## Find distance from under or overshooting ports and select uids near those ports
            rw_one_dist = field_dist - first_rw_pos
            rw_two_dist = field_dist - second_rw_pos
            bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
            rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
            rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
            sub_uids = neural_data['unit_id'][active_cells]
            sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

            for idx, d in enumerate(days_of_int):
                if d != reference_day:
                    S = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{d}.nc'))[data_type]
                    sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()) ## select the equivalent cell from the ref day
                    neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                    velocity_thresh=velocity_thresh)
                    ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                    population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                    active_cells = np.sum(population_activity, axis=0) != 0
                    population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                    tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                    tuning_curves = tuning_curves.T ## cells x spatial bin
                    ## Get where the peak of the tuning curve is for each cell
                    field_dist = pc.place_field_peak(tuning_curves, bins)

                    ## Get reward positions
                    reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                    reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                    ## Make Reward 1 the first rewarding port the mouse got water from
                    first_rew = sdata['lick_port'][sdata['water']].values[0]
                    if first_rew == sdata.attrs['reward_one']:
                        first_rw_pos = reward_one_pos 
                        second_rw_pos = reward_two_pos
                    else:
                        first_rw_pos = reward_two_pos 
                        second_rw_pos = reward_one_pos

                    ## Find distance from reward locations
                    ## Negative values would be after the reward port because linear position decreases across a trial
                    distance_from_both = np.zeros((field_dist.shape[0], 2))
                    distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
                    distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
                    min_rw_distance = np.min(distance_from_both, axis=1)
                    min_rw_distance = min_rw_distance * conversion ## convert to centimeters

                    ## Get firing rate across the session while running in the correct direction
                    act_bin = ctn.bin_activity(neural_data[active_cells].values, bin_size_seconds=fr_bin_size, func=np.mean) ## select active sells
                    avg_act = np.mean(act_bin, axis=1) / fr_bin_size ## convert to Hz for any bin size

                    for idx in np.arange(0, min_rw_distance.shape[0]):
                        cell_dict['mouse'].append(mouse)
                        cell_dict['group'].append(S.attrs['group'])
                        cell_dict['sex'].append(S.attrs['sex'])
                        cell_dict['day'].append(d)
                        cell_dict['cell_type'].append(port_type)
                        cell_dict['unit_id'].append(sub_uids.values[idx])
                        cell_dict['min_rw_distance'].append(min_rw_distance[idx])
                        cell_dict['spatial_info'].append(sdata['skaggs_info'].values[idx])
                        cell_dict['firing_rate'].append(avg_act[idx])
                        cell_dict['odd_even'].append(sdata['odd_even'].values[idx])
                        cell_dict['first_second'].append(sdata['first_second'].values[idx])
non_rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
## Combine both dataframes for plotting
combined_df = pd.concat([rw_dist_df, non_rw_dist_df])
avg_data_mouse = combined_df.groupby(['cell_type', 'mouse', 'group'], as_index=False).agg({'min_rw_distance': 'mean', 'spatial_info': 'mean', 'firing_rate': 'mean',
                                                                                           'odd_even': 'mean', 'first_second': 'mean'})
avg_data = avg_data_mouse.groupby(['cell_type', 'group'], as_index=False).agg({'min_rw_distance': ['mean', 'sem'], 'spatial_info': ['mean', 'sem'],
                                                                               'firing_rate': ['mean', 'sem'], 'odd_even': ['mean', 'sem'], 'first_second': ['mean', 'sem']})

In [ ]:
## Plot reward distance for cells from the reward area or from the undershooting/overshooting port for both groups
fig = pf.custom_graph_template(x_title='', y_title='Reward Distance (cm)')

for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = avg_data[avg_data['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cell_type'], y=gdata['min_rw_distance']['mean'], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group,
                               error_y=dict(type='data', array=gdata['min_rw_distance']['sem'], thickness=2.5)))
fig.show()

In [ ]:
## Plot spatial information for cells from the reward area or from the undershooting/overshooting port for both groups
fig = pf.custom_graph_template(x_title='', y_title='Spatial Info (bits/event)')

for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = avg_data[avg_data['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cell_type'], y=gdata['spatial_info']['mean'], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=8, name=group,
                               error_y=dict(type='data', array=gdata['spatial_info']['sem'])))
fig.show()

In [ ]:
## Plot event rate for cells from the reward area or from the undershooting/overshooting port for both groups
fig = pf.custom_graph_template(x_title='', y_title='Event Rate (Hz)')

for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = avg_data[avg_data['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cell_type'], y=gdata['firing_rate']['mean'], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group,
                               error_y=dict(type='data', array=gdata['firing_rate']['sem'], thickness=2.5)))
fig.show()

In [ ]:
## Plot stability for cells from the reward area or from the undershooting/overshooting port for both groups
fig = pf.custom_graph_template(x_title='', y_title='Odd-Even Stability')

for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = avg_data[avg_data['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cell_type'], y=gdata['odd_even']['mean'], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group,
                               error_y=dict(type='data', array=gdata['odd_even']['sem'], thickness=2.5)))
fig.show()

In [ ]:
## Plot stability for cells from the reward area or from the undershooting/overshooting port for both groups
fig = pf.custom_graph_template(x_title='', y_title='First-Second Stability')

for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = avg_data[avg_data['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cell_type'], y=gdata['first_second']['mean'], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group,
                               error_y=dict(type='data', array=gdata['first_second']['sem'], thickness=2.5)))
fig.show()

In [ ]:
## Of cells from around reward area or undershooting/overshooting area, what was their reward distance on the day before the reference day?
avg = avg_data_mouse.groupby(['cell_type'], as_index=False).agg({'min_rw_distance': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Reward Distance (cm)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['min_rw_distance']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['min_rw_distance']['sem'], thickness=2.5)))
for mouse in combined_df['mouse'].unique():
    mdata = avg_data_mouse[avg_data_mouse['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['min_rw_distance'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.show()
fig.write_image(pjoin(fig_path, f'rw_area_vs_{port_type}_days{days_of_int}_rw_distance.png'), width=400, height=500)

In [ ]:
## Of cells from around reward area or undershooting/overshooting area, what was their spatial information on the day before the reference day?
avg = avg_data_mouse.groupby(['cell_type'], as_index=False).agg({'min_rw_distance': ['mean', 'sem'], 'spatial_info': ['mean', 'sem'], 'firing_rate': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Spatial Info (bits/event)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['spatial_info']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['spatial_info']['sem'], thickness=2.5)))
for mouse in combined_df['mouse'].unique():
    mdata = avg_data_mouse[avg_data_mouse['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['spatial_info'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.show()
fig.write_image(pjoin(fig_path, f'rw_area_vs_{port_type}_days{days_of_int}_spatial_info.png'), width=400, height=500)

In [ ]:
## Of cells from around reward area or undershooting/overshooting area, what was their firing rate on the day before the reference day?
avg = avg_data_mouse.groupby(['cell_type'], as_index=False).agg({'min_rw_distance': ['mean', 'sem'], 'spatial_info': ['mean', 'sem'], 'firing_rate': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Event Rate (Hz)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['firing_rate']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['firing_rate']['sem'], thickness=2.5)))
for mouse in combined_df['mouse'].unique():
    mdata = avg_data_mouse[avg_data_mouse['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['firing_rate'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 0.025])
fig.show()
fig.write_image(pjoin(fig_path, f'rw_area_vs_{port_type}_days{days_of_int}_event_rate.png'), width=400, height=500)

In [ ]:
## Of cells from around reward area or undershooting/overshooting area, what was their first-second stability on the day before the reference day?
avg = avg_data_mouse.groupby(['cell_type'], as_index=False).agg({'first_second': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='First-Second Stability', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['first_second']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['first_second']['sem'], thickness=2.5)))
for mouse in combined_df['mouse'].unique():
    mdata = avg_data_mouse[avg_data_mouse['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['first_second'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[-0.1, 1], dtick=0.2)
fig.show()
fig.write_image(pjoin(fig_path, f'rw_area_vs_{port_type}_days{days_of_int}_first_second.png'), width=400, height=500)

In [ ]:
## Of cells from around reward area or undershooting/overshooting area, what was their odd-even stability on the day before the reference day?
avg = avg_data_mouse.groupby(['cell_type'], as_index=False).agg({'odd_even': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Odd-Even Stability', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['odd_even']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['odd_even']['sem'], thickness=2.5)))
for mouse in combined_df['mouse'].unique():
    mdata = avg_data_mouse[avg_data_mouse['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['odd_even'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[-0.1, 1], dtick=0.2)
fig.show()
fig.write_image(pjoin(fig_path, f'rw_area_vs_{port_type}_days{days_of_int}_odd_even.png'), width=400, height=500)

### Compare activity of cells from the reward area to an equal sized random subset across the track.

In [ ]:
## Settings
only_running = True 
correct_dir = True
days_of_int = [11, 16] ## make reference day your second day. Asking from the cells on the second day in the list, what were they doing on the first day in the list
reference_day = 16
crossreg_str = None
cell_type = 'all_cells'
num_iterations = 50 ## number of times the code will pick a random subset of cells
# excluded_mice = ['mc46'] ## if comparing to day 11, add mc46 to list, add mc56 if comparing day 1, mc52 if comparing day 4
excluded_mice = ['mc46']

In [ ]:
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': [], 
             'firing_rate': [], 'odd_even': [], 'first_second': []}

for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        fpath = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(fpath)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(fpath, f'{mouse}/{data_type}')
                crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
                mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)
                
                date_list = []
                for d in days_of_int:
                    session = f'{mouse}_{data_type}_{d}.nc'
                    S = xr.load_dataset(pjoin(mpath, session))[data_type]
                    date_list.append(S.attrs['date'])
                ## Select shared cells
                shared_cells = mappings[date_list].dropna().reset_index(drop=True)
                ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
                sdata = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{reference_day}.nc'))
                sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
                sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :]
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                else:
                    pass

                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)

                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos

                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos
                bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
                rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
                sub_uids = neural_data['unit_id'][active_cells]
                sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

                for idx, d in enumerate(days_of_int):
                    if d != reference_day:
                        S = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{d}.nc'))[data_type]
                        sdata = S.sel(unit_id=shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()) ## select the equivalent cell from the ref day
                        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                        velocity_thresh=velocity_thresh)
                        ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                        active_cells = np.sum(population_activity, axis=0) != 0
                        population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                        tuning_curves = tuning_curves.T ## cells x spatial bin
                        ## Get where the peak of the tuning curve is for each cell
                        field_dist = pc.place_field_peak(tuning_curves, bins)

                        ## Get reward positions
                        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                        ## Make Reward 1 the first rewarding port the mouse got water from
                        first_rew = sdata['lick_port'][sdata['water']].values[0]
                        if first_rew == sdata.attrs['reward_one']:
                            first_rw_pos = reward_one_pos 
                            second_rw_pos = reward_two_pos
                        else:
                            first_rw_pos = reward_two_pos 
                            second_rw_pos = reward_one_pos

                        ## Find distance from reward locations
                        ## Negative values would be after the reward port because linear position decreases across a trial
                        distance_from_both = np.zeros((field_dist.shape[0], 2))
                        distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
                        distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
                        min_rw_distance = np.min(distance_from_both, axis=1)
                        min_rw_distance = min_rw_distance * conversion ## convert to centimeters

                        ## Get firing rate across the session while running in the correct direction
                        act_bin = ctn.bin_activity(neural_data[active_cells].values, bin_size_seconds=fr_bin_size, func=np.mean) ## select active sells
                        avg_act = np.mean(act_bin, axis=1) / fr_bin_size ## convert to Hz for any bin size

                        for idx in np.arange(0, min_rw_distance.shape[0]):
                            cell_dict['mouse'].append(mouse)
                            cell_dict['group'].append(S.attrs['group'])
                            cell_dict['sex'].append(S.attrs['sex'])
                            cell_dict['day'].append(d)
                            cell_dict['cell_type'].append('Reward Area')
                            cell_dict['unit_id'].append(sub_uids.values[idx])
                            cell_dict['min_rw_distance'].append(min_rw_distance[idx])
                            cell_dict['spatial_info'].append(sdata['skaggs_info'].values[idx])
                            cell_dict['firing_rate'].append(avg_act[idx])
                            cell_dict['odd_even'].append(sdata['odd_even'].values[idx])
                            cell_dict['first_second'].append(sdata['first_second'].values[idx])
rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
cell_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'sim': [], 'cell_type': [], 'unit_id': [], 'min_rw_distance': [], 'spatial_info': [], 
             'firing_rate': [], 'odd_even': [], 'first_second': []}

for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        fpath = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(fpath)):
            if mouse in excluded_mice:
                pass 
            else:
                mpath = pjoin(fpath, f'{mouse}/{data_type}')
                crossreg_path = f'../../../CircleTrack/{project_folder[0]}/{experiment}/output/cross_registration_results/circletrack_data/{mouse}'
                mappings = pd.read_pickle(pjoin(crossreg_path, f'mappings_{centroid_distance}_{crossreg_str}.pkl'))['session'].reset_index(drop=True)
                
                date_list = []
                for d in days_of_int:
                    session = f'{mouse}_{data_type}_{d}.nc'
                    S = xr.load_dataset(pjoin(mpath, session))[data_type]
                    date_list.append(S.attrs['date'])
                ## Select shared cells
                shared_cells = mappings[date_list].dropna().reset_index(drop=True)
                ref_idx = np.where(np.array(days_of_int) == reference_day)[0][0]
                sdata = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{reference_day}.nc'))
                sdata = S.sel(unit_id=shared_cells[date_list[ref_idx]].to_numpy()) ## use the reference day to index date list
                sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :]
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                else:
                    pass

                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)

                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos

                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos
                bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
                rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
                sub_uids = neural_data['unit_id'][active_cells]
                sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

                for idx, d in enumerate(days_of_int):
                    if d != reference_day:
                        rs = RandomState(MT19937(SeedSequence(24601)))
                        S = xr.load_dataset(pjoin(mpath, f'{mouse}_{data_type}_{d}.nc'))[data_type]
                        for it in np.arange(num_iterations):
                            ## remove the reward over-rep cells from unit_ids so they aren't included in the subsetting
                            num_cells = shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy().shape[0]
                            sdata = S[~np.isin(S['unit_id'].values, shared_cells[S.attrs['date']][shared_cells[date_list[ref_idx]].isin(sub_uids.values)].to_numpy()), :]
                            random_subset = rs.randint(low=0, high=sdata.shape[0], size=num_cells) ## select the same number of indices as number of cells from the reward area
                            sub_data = sdata[random_subset, :]
                            neural_data, position_data = ctn.subset_correct_dir_and_running(sub_data, correct_dir=correct_dir, only_running=only_running, 
                                                                                            velocity_thresh=velocity_thresh)
                            ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                            population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                            active_cells = np.sum(population_activity, axis=0) != 0
                            population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                            tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                            tuning_curves = tuning_curves.T ## cells x spatial bin
                            ## Get where the peak of the tuning curve is for each cell
                            field_dist = pc.place_field_peak(tuning_curves, bins)

                            ## Get reward positions
                            reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                            reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                            ## Make Reward 1 the first rewarding port the mouse got water from
                            first_rew = sdata['lick_port'][sdata['water']].values[0]
                            if first_rew == sdata.attrs['reward_one']:
                                first_rw_pos = reward_one_pos 
                                second_rw_pos = reward_two_pos
                            else:
                                first_rw_pos = reward_two_pos 
                                second_rw_pos = reward_one_pos

                            ## Find distance from reward locations
                            ## Negative values would be after the reward port because linear position decreases across a trial
                            distance_from_both = np.zeros((field_dist.shape[0], 2))
                            distance_from_both[:, 0] = abs(field_dist - first_rw_pos)
                            distance_from_both[:, 1] = abs(field_dist - second_rw_pos)
                            min_rw_distance = np.min(distance_from_both, axis=1)
                            min_rw_distance = min_rw_distance * conversion ## convert to centimeters

                            ## Get firing rate across the session while running in the correct direction
                            act_bin = ctn.bin_activity(neural_data[active_cells].values, bin_size_seconds=fr_bin_size, func=np.mean) ## select active sells
                            avg_act = np.mean(act_bin, axis=1) / fr_bin_size ## convert to Hz for any bin size

                            for idx in np.arange(0, min_rw_distance.shape[0]):
                                cell_dict['mouse'].append(mouse)
                                cell_dict['group'].append(S.attrs['group'])
                                cell_dict['sex'].append(S.attrs['sex'])
                                cell_dict['day'].append(d)
                                cell_dict['sim'].append(it)
                                cell_dict['cell_type'].append('Random')
                                cell_dict['unit_id'].append(sub_data['unit_id'].values[idx])
                                cell_dict['min_rw_distance'].append(min_rw_distance[idx])
                                cell_dict['spatial_info'].append(sub_data['skaggs_info'].values[idx])
                                cell_dict['firing_rate'].append(avg_act[idx])
                                cell_dict['odd_even'].append(sub_data['odd_even'].values[idx])
                                cell_dict['first_second'].append(sub_data['first_second'].values[idx])
non_rw_dist_df = pd.DataFrame(cell_dict)

In [ ]:
## Combine both dataframes for plotting
combined_df = pd.concat([rw_dist_df, non_rw_dist_df])
avg_data_mouse = combined_df.groupby(['cell_type', 'mouse', 'group'], as_index=False).agg({'min_rw_distance': 'mean', 'spatial_info': 'mean', 'firing_rate': 'mean',
                                                                                           'odd_even': 'mean', 'first_second': 'mean'})
avg_data = avg_data_mouse.groupby(['cell_type', 'group'], as_index=False).agg({'min_rw_distance': ['mean', 'sem'], 'spatial_info': ['mean', 'sem'],
                                                                               'firing_rate': ['mean', 'sem'], 'odd_even': ['mean', 'sem'], 'first_second': ['mean', 'sem']})

In [ ]:
## Of cells from around reward area or random, what was their reward distance on the day before the reference day?
g = 'Multi-context'
if g is not None:
    sub_d = avg_data_mouse[avg_data_mouse['group'] == g]
else:
    sub_d = avg_data_mouse.copy()
avg = sub_d.groupby(['cell_type', 'group'], as_index=False).agg({'min_rw_distance': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Reward Distance (cm)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['min_rw_distance']['mean'], mode='markers', marker_color=ce_colors_dict[g],
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['min_rw_distance']['sem'], thickness=2.5)))
for mouse in sub_d['mouse'].unique():
    mdata = sub_d[sub_d['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['min_rw_distance'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 75])
fig.show()
fig.write_image(pjoin(fig_path, f'{g}_rw_area_vs_random_subset_days{days_of_int}_rw_distance.png'), width=400, height=500)

## Perform paired t-test
ttest_rel(sub_d['min_rw_distance'][sub_d['cell_type'] == 'Random'], sub_d['min_rw_distance'][sub_d['cell_type'] == 'Reward Area'])

In [ ]:
## Of cells from around reward area or random, what was their spatial info on the day before the reference day?
g = 'Multi-context'
if g is not None:
    sub_d = avg_data_mouse[avg_data_mouse['group'] == g]
else:
    sub_d = avg_data_mouse.copy()
avg = sub_d.groupby(['cell_type', 'group'], as_index=False).agg({'spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Spatial Info (bits/event)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['spatial_info']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['spatial_info']['sem'], thickness=2.5)))
for mouse in sub_d['mouse'].unique():
    mdata = sub_d[sub_d['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['spatial_info'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 6.6]) # 0 to 6.6 normally, 0 to 11 if comparing between two days in the same context
fig.show()
fig.write_image(pjoin(fig_path, f'{g}_rw_area_vs_random_subset_days{days_of_int}_spatial_info.png'), width=400, height=500)

## Perform paired t-test
ttest_rel(sub_d['spatial_info'][sub_d['cell_type'] == 'Random'], sub_d['spatial_info'][sub_d['cell_type'] == 'Reward Area'])

In [ ]:
## Of cells from around reward area or random, what was their firing rate on the day before the reference day?
g = 'Two-context'
if g is not None:
    sub_d = avg_data_mouse[avg_data_mouse['group'] == g]
else:
    sub_d = avg_data_mouse.copy()
avg = sub_d.groupby(['cell_type', 'group'], as_index=False).agg({'firing_rate': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Event Rate (Hz)', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['firing_rate']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['firing_rate']['sem'], thickness=2.5)))
for mouse in sub_d['mouse'].unique():
    mdata = sub_d[sub_d['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['firing_rate'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 0.012])
fig.show()
fig.write_image(pjoin(fig_path, f'{g}_rw_area_vs_random_subset_days{days_of_int}_firing_rate.png'), width=400, height=500)

## Perform paired t-test
ttest_rel(sub_d['firing_rate'][sub_d['cell_type'] == 'Random'], sub_d['firing_rate'][sub_d['cell_type'] == 'Reward Area'])

In [ ]:
## Of cells from around reward area or random, what was their first-second stability on the day before the reference day?
g = 'Two-context'
if g is not None:
    sub_d = avg_data_mouse[avg_data_mouse['group'] == g]
else:
    sub_d = avg_data_mouse.copy()
avg = sub_d.groupby(['cell_type', 'group'], as_index=False).agg({'first_second': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='First-Second Stability', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['first_second']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['first_second']['sem'], thickness=2.5)))
for mouse in sub_d['mouse'].unique():
    mdata = sub_d[sub_d['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['first_second'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 1], dtick=0.2)
fig.show()
fig.write_image(pjoin(fig_path, f'{g}_rw_area_vs_random_subset_days{days_of_int}_first_second.png'), width=400, height=500)

## Perform paired t-test
ttest_rel(sub_d['first_second'][sub_d['cell_type'] == 'Random'], sub_d['first_second'][sub_d['cell_type'] == 'Reward Area'])

In [ ]:
## Of cells from around reward area or random, what was their odd-even stability on the day before the reference day?
g = 'Two-context'
if g is not None:
    sub_d = avg_data_mouse[avg_data_mouse['group'] == g]
else:
    sub_d = avg_data_mouse.copy()
avg = sub_d.groupby(['cell_type', 'group'], as_index=False).agg({'odd_even': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Odd-Even Stability', width=400)
fig.add_trace(go.Scattergl(x=avg['cell_type'], y=avg['odd_even']['mean'], mode='markers', marker_color='darkgrey',
                            marker=dict(line=dict(width=1.5, color='black')), marker_size=9, showlegend=False,
                            error_y=dict(type='data', array=avg['odd_even']['sem'], thickness=2.5)))
for mouse in sub_d['mouse'].unique():
    mdata = sub_d[sub_d['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['cell_type'], y=mdata['odd_even'], mode='lines', line_color=ce_colors_dict[mdata['group'].unique()[0]], 
                               line_width=0.8, name=mouse, showlegend=False))
fig.update_yaxes(range=[0, 1], dtick=0.2)
fig.show()
fig.write_image(pjoin(fig_path, f'{g}_rw_area_vs_random_subset_days{days_of_int}_odd_even.png'), width=400, height=500)

## Perform paired t-test
ttest_rel(sub_d['odd_even'][sub_d['cell_type'] == 'Random'], sub_d['odd_even'][sub_d['cell_type'] == 'Reward Area'])